# Tutorial: The Great Algorithm Comparison

> **Type:** Tutorial — follow along step by step. Your only job is to make it work.  
> **Time:** ~2 hours  
> **Prerequisites:** Module 06 (Python ecosystem), Module 10 (DS lifecycle)  
> **What you'll build:** A side-by-side comparison of five classifiers on real medical data, including decision boundary visualisations and a performance benchmark chart.

---

## What We're Building

We're going to answer a single question: **which algorithm works best for classifying breast tumours as malignant or benign?** But the real point isn't the answer — it's the *process*. You'll see that different algorithms make very different assumptions, produce very different decision boundaries, and have very different speed characteristics. By the end, you'll have the intuition to pick a starting algorithm for any classification problem.

## Step 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Setup complete.')

## Step 1: Load and Understand the Data

The breast cancer dataset is a classic: 569 tumour samples, each described by 30 numeric features (radius, texture, smoothness, etc.), each labelled as malignant (0) or benign (1). It's real clinical data.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')  # 0=malignant, 1=benign

print(f'Shape: {X.shape}')
print(f'Features: {list(X.columns[:5])} ... ({X.shape[1]} total)')
print(f'\nClass distribution:')
print(y.value_counts().rename({0: 'Malignant', 1: 'Benign'}))
print(f'\nClass balance: {y.mean():.1%} benign')
X.describe().round(2)

In [ ]:
# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

> **What just happened?** We loaded a real dataset with 30 features. `stratify=y` ensures the train/test split has the same class ratio as the full dataset — important for imbalanced data.

## Step 2: Define the Five Classifiers

Notice something important: **every sklearn classifier has the same API**. `.fit()`, `.predict()`, `.predict_proba()`. Once you know the pattern, switching algorithms is trivial.

In [ ]:
# We use Pipelines so scaling is applied consistently
# (KNN and Logistic Regression need scaled features; trees don't but it doesn't hurt)
classifiers = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
    ]),
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(random_state=RANDOM_SEED))
    ]),
    'K-Nearest Neighbours': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GaussianNB())
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=100))
    ]),
}

print(f'{len(classifiers)} classifiers defined.')

## Step 3: Train and Benchmark All Five

In [ ]:
results = []

for name, clf in classifiers.items():
    # Training time
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    train_time = (time.perf_counter() - t0) * 1000  # ms
    
    # Prediction time
    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    pred_time = (time.perf_counter() - t0) * 1000  # ms
    
    # Cross-validated accuracy (5-fold on training set — more honest than test set)
    cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')
    
    test_acc = accuracy_score(y_test, y_pred)
    
    results.append({
        'Algorithm': name,
        'Test Accuracy': test_acc,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Train Time (ms)': train_time,
        'Predict Time (ms)': pred_time,
    })
    print(f'{name:<25} test={test_acc:.3f}  cv={cv_scores.mean():.3f}±{cv_scores.std():.3f}  train={train_time:.1f}ms')

results_df = pd.DataFrame(results).sort_values('CV Mean', ascending=False)
print('\nDone.')

> **What just happened?** We trained all five classifiers and evaluated them two ways: on the held-out test set, and via 5-fold cross-validation on the training set. CV mean is more reliable because it averages over multiple splits. The std tells you how stable the estimate is.

## Step 4: Visualise the Benchmark

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colours = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2', '#59A14F']
algo_names = results_df['Algorithm'].tolist()

# --- Accuracy ---
ax = axes[0]
bars = ax.barh(algo_names, results_df['CV Mean'], xerr=results_df['CV Std'],
               color=colours, capsize=4, alpha=0.85)
ax.set_xlabel('5-fold CV Accuracy')
ax.set_title('Accuracy (CV Mean ± Std)', fontweight='bold')
ax.set_xlim(0.88, 1.0)
for bar, val in zip(bars, results_df['CV Mean']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# --- Training time ---
ax = axes[1]
ax.barh(algo_names, results_df['Train Time (ms)'], color=colours, alpha=0.85)
ax.set_xlabel('Training time (ms)')
ax.set_title('Training Time', fontweight='bold')
ax.set_xscale('log')

# --- Prediction time ---
ax = axes[2]
ax.barh(algo_names, results_df['Predict Time (ms)'], color=colours, alpha=0.85)
ax.set_xlabel('Prediction time (ms)')
ax.set_title('Prediction Time (test set)', fontweight='bold')
ax.set_xscale('log')

plt.tight_layout()
plt.savefig('algorithm_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print(results_df[['Algorithm', 'CV Mean', 'CV Std', 'Train Time (ms)', 'Predict Time (ms)']].to_string(index=False))

## Step 5: Visualise Decision Boundaries

To visualise decision boundaries we need 2D data. We'll pick the two most informative features and train each classifier on just those two.

In [ ]:
# Use 'mean radius' and 'mean texture' — two clearly informative features
feature_1, feature_2 = 'mean radius', 'mean texture'
X_2d = X[[feature_1, feature_2]]

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# Re-define without Pipeline (DecisionBoundaryDisplay needs the raw classifier + scaler handled separately)
classifiers_2d = {
    'Logistic Regression':  LogisticRegression(random_state=RANDOM_SEED),
    'Decision Tree':        DecisionTreeClassifier(random_state=RANDOM_SEED),
    'K-Nearest Neighbours': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':          GaussianNB(),
    'Random Forest':        RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=100),
}

scaler_2d = StandardScaler()
X_train_2d_sc = scaler_2d.fit_transform(X_train_2d)
X_test_2d_sc  = scaler_2d.transform(X_test_2d)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
cmap = mcolors.ListedColormap(['#FADADD', '#DAE8FC'])  # light red, light blue

for ax, (name, clf) in zip(axes, classifiers_2d.items()):
    clf.fit(X_train_2d_sc, y_train_2d)
    acc_2d = accuracy_score(y_test_2d, clf.predict(X_test_2d_sc))
    
    DecisionBoundaryDisplay.from_estimator(
        clf, X_train_2d_sc, ax=ax, response_method='predict',
        cmap=cmap, alpha=0.7
    )
    # Scatter actual points
    scatter_cols = ['#E15759', '#4E79A7']  # red=malignant, blue=benign
    for cls, col in enumerate(scatter_cols):
        mask = y_train_2d == cls
        ax.scatter(X_train_2d_sc[mask, 0], X_train_2d_sc[mask, 1],
                   color=col, s=15, alpha=0.5, label=['Malignant', 'Benign'][cls])
    
    ax.set_title(f'{name}\nacc={acc_2d:.3f}', fontsize=9, fontweight='bold')
    ax.set_xlabel(feature_1, fontsize=7)
    ax.set_ylabel(feature_2, fontsize=7)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7, loc='upper right')
plt.suptitle('Decision Boundaries — 2 Features Only (radius vs texture)', 
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** The boundaries reveal each algorithm's personality: Logistic Regression draws a straight line. Decision Tree makes rectangular cuts. KNN produces a jagged, local boundary. Naive Bayes draws an ellipse. Random Forest blends many trees into a smoother nonlinear boundary.

## Step 6: Inspect the Best Model's Report

In [ ]:
# Full classification report for the best model (Random Forest on all 30 features)
best_clf = classifiers['Random Forest']
y_pred_best = best_clf.predict(X_test)

print('=== Random Forest — Full Classification Report ===')
print(classification_report(y_test, y_pred_best, target_names=['Malignant', 'Benign']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: The No Free Lunch Demonstration

In [ ]:
# Create a dataset where a simple linear classifier wins:
# two blobs perfectly separated by a line
from sklearn.datasets import make_blobs, make_circles

X_blobs, y_blobs = make_blobs(n_samples=300, centers=2, random_state=RANDOM_SEED, cluster_std=1.5)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=RANDOM_SEED)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

datasets = [
    ('Linear-separable blobs', X_blobs, y_blobs),
    ('Non-linear circles', X_circles, y_circles),
]
models = [
    ('Logistic Regression', LogisticRegression()),
    ('Random Forest', RandomForestClassifier(random_state=RANDOM_SEED)),
]

cmap2 = mcolors.ListedColormap(['#FADADD', '#DAE8FC'])

for row, (ds_name, X_ds, y_ds) in enumerate(datasets):
    sc = StandardScaler()
    X_ds_sc = sc.fit_transform(X_ds)
    X_tr, X_te, y_tr, y_te = train_test_split(X_ds_sc, y_ds, test_size=0.3, random_state=RANDOM_SEED)
    
    for col, (model_name, model) in enumerate(models):
        model.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, model.predict(X_te))
        
        ax = axes[row][col]
        DecisionBoundaryDisplay.from_estimator(model, X_ds_sc, ax=ax, 
                                                response_method='predict', cmap=cmap2, alpha=0.6)
        ax.scatter(X_ds_sc[:, 0], X_ds_sc[:, 1], c=y_ds, cmap='bwr', s=20, alpha=0.7, edgecolors='k', linewidths=0.3)
        ax.set_title(f'{ds_name}\n{model_name} — acc={acc:.3f}', fontweight='bold', fontsize=10)

plt.suptitle('No Free Lunch: the best algorithm depends on the data shape', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('no_free_lunch.png', dpi=150, bbox_inches='tight')
plt.show()

---

## What You Built

You trained five classifiers on a real medical dataset, benchmarked them on accuracy and speed, visualised their decision boundaries, and demonstrated the No Free Lunch theorem empirically — Logistic Regression wins on linearly separable data; Random Forest wins on the circles. You also saw that the sklearn API is identical across all five algorithms: `.fit()`, `.predict()`, `.predict_proba()`.

## What's Next

- Read `explanation.md` if you haven't — the bias-variance section will make the decision boundary visualisations much clearer.
- Open `howto.md` for quick recipes on choosing and training classifiers.
- **Extension challenge:** Add an SVM (`sklearn.svm.SVC`) to the comparison. How does it compare in accuracy and speed?

**Next module:** `12_ensemble_methods` — where you learn why combining many weak models almost always beats one strong model.

---

**AWS SageMaker version:** All sklearn classifiers run identically in a SageMaker Studio notebook. For production scale, use SageMaker's built-in XGBoost or SKLearn containers — see `26_cloud_ml_sagemaker` for setup.